# Module 02 — Lesson 2: Prompt Engineering Techniques with Google Gemini
Google Gemini Companion Implementation • Dual-API Parity with `google-genai` SDK

---

## 🎯 Dual-API Parity: Claude & Gemini Prompt Engineering
While Anthropic Claude natively optimizes for XML tags (`<instructions>`, `<document>`), **Google Gemini (Gemini 2.5 Flash / 1.5 Flash)** excels when prompts leverage:
1. **Explicit `system_instruction` Configuration**: Decoupling the high-level role/persona from the user content turn via `types.GenerateContentConfig(system_instruction=...)`.
2. **Structured Section Delimiters**: Utilizing clean Markdown headers (`## Section`) or tagged structural delimiters (`<context>`, `<guidelines>`).
3. **Structured Pydantic Schemas**: Native JSON schema enforcement with `response_mime_type="application/json"` and `response_schema=...`.
4. **Few-Shot In-Context Exemplars**: Supplying demonstration turns or inline `<example>` blocks to steer format compliance and handle nuance.

In this companion lab, we apply the exact same **Athletic Meal Planning** benchmark suite to measure the impact of systematic prompt engineering on Gemini.



In [ ]:
# Step 1: Environment Setup & Google GenAI Client Initialization
import os
import re
import json
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

load_dotenv()

# Initialize Google GenAI client
client = genai.Client()

# Recommended Gemini production model
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

print(f"✅ Google GenAI Client Initialized. Target Model: {GEMINI_MODEL}")



In [ ]:
# Helper Functions: Gemini Prompt Execution
def prompt_gemini(prompt_text, system_instruction=None, temperature=0.2, response_schema=None):
    """Execute a Google Gemini content generation call using modern google-genai SDK."""
    config_params = {
        "temperature": temperature,
    }
    if system_instruction:
        config_params["system_instruction"] = system_instruction
    if response_schema:
        config_params["response_mime_type"] = "application/json"
        config_params["response_schema"] = response_schema
        
    config = types.GenerateContentConfig(**config_params)
    
    try:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt_text,
            config=config,
        )
        return response.text
    except Exception as e:
        # Fallback to gemini-1.5-flash if the primary model is unavailable
        response = client.models.generate_content(
            model="gemini-1.5-flash",
            contents=prompt_text,
            config=config,
        )
        return response.text



---
## Step 2: The Benchmark Evaluation Test Suite
We use the exact same 5 calibrated athletic nutrition scenarios to maintain mathematical parity with our Claude benchmark.



In [ ]:
# Shared Golden Test Dataset
test_dataset = [
    {
        "scenario": "Endurance Runner with Lactose Intolerance",
        "prompt_inputs": {
            "height": "178 cm",
            "weight": "68 kg",
            "goal": "Marathon training endurance with high carbohydrate needs",
            "restrictions": "Lactose intolerant (dairy-free)"
        },
        "solution_criteria": [
            "Includes daily caloric total appropriate for endurance athlete (~2800-3200 kcal)",
            "Lists macronutrient grams (protein, fat, carbohydrates)",
            "Specifies exact meal timing (e.g. 7:00 AM breakfast, 12:30 PM lunch, etc.)",
            "Contains strictly zero dairy/lactose ingredients"
        ]
    },
    {
        "scenario": "Strength Athlete with Hypercholesterolemia",
        "prompt_inputs": {
            "height": "182 cm",
            "weight": "88 kg",
            "goal": "Lean muscle hypertrophy without raising LDL cholesterol",
            "restrictions": "High cholesterol (strict low saturated fat, zero trans fat)"
        },
        "solution_criteria": [
            "Includes daily caloric total appropriate for hypertrophy (~2900-3300 kcal)",
            "Lists macronutrient grams with high lean protein (>= 160g)",
            "Specifies exact meal timing across at least 4 meals/snacks",
            "Restricts saturated fats, egg yolks, and high-cholesterol meats"
        ]
    },
    {
        "scenario": "Weight-Class Fighter Cutting Weight (Celiac Disease)",
        "prompt_inputs": {
            "height": "172 cm",
            "weight": "74 kg",
            "goal": "Cutting 4kg for upcoming competition while preserving explosive power",
            "restrictions": "Celiac disease (strict gluten-free)"
        },
        "solution_criteria": [
            "Includes moderate caloric deficit (~2000-2200 kcal)",
            "Specifies exact portion sizes in grams for all foods",
            "Contains strictly zero gluten (no wheat, barley, rye)",
            "Distributes protein evenly across pre- and post-workout windows"
        ]
    },
    {
        "scenario": "Vegetarian Swimmer with Budget Constraints",
        "prompt_inputs": {
            "height": "185 cm",
            "weight": "76 kg",
            "goal": "High daily swimming volume recovery",
            "restrictions": "Strict vegetarian (no meat/poultry/fish), low-cost ingredients"
        },
        "solution_criteria": [
            "Includes daily caloric total (~3000-3400 kcal)",
            "Provides complete plant proteins (tofu, legumes, lentils, oats)",
            "Mentions affordable pantry-friendly food choices",
            "Lists specific meal timing schedule"
        ]
    },
    {
        "scenario": "Diabetic Master CrossFitter",
        "prompt_inputs": {
            "height": "168 cm",
            "weight": "65 kg",
            "goal": "High-intensity metabolic conditioning without glucose spikes",
            "restrictions": "Type 2 Diabetes (low glycemic index, zero refined sugars)"
        },
        "solution_criteria": [
            "Specifies exact meal schedule and snack timings",
            "Lists low-GI complex carbohydrate sources and gram portions",
            "Excludes all refined sugars, juices, and high-glycemic foods",
            "Includes complete macronutrient and calorie breakdown"
        ]
    }
]

print(f"✅ Loaded {len(test_dataset)} test scenarios for Gemini benchmarking.")



In [ ]:
# Structured Model Grader with Pydantic & Gemini
class EvaluationResult(BaseModel):
    score: float = Field(description="Numerical grade from 1.0 to 10.0 based on criteria fulfillment")
    passed_criteria: list[str] = Field(description="List of criteria that the solution satisfied")
    failed_criteria: list[str] = Field(description="List of criteria that the solution failed")
    reasoning: str = Field(description="Concise 1-2 sentence rationale for the score")

def grade_solution_gemini(test_case, solution_output):
    """Grade candidate output using Gemini with structured Pydantic schema validation."""
    criteria_str = "\n".join([f"- {c}" for c in test_case["solution_criteria"]])
    prompt_inputs_str = json.dumps(test_case["prompt_inputs"], indent=2)

    eval_prompt = f"""
You are an expert sports performance nutritionist and strict AI evaluation judge.
Evaluate the following candidate meal plan against the scenario requirements and criteria with extreme rigor.

Scenario: {test_case["scenario"]}

Athlete Profile Inputs:
{prompt_inputs_str}

Candidate Solution to Evaluate:
{solution_output}

Evaluation Criteria:
{criteria_str}

Scoring Rubric (1.0 to 10.0):
- 1-3: Critical failure (violates dietary restriction, missing calories/macros, completely uncalibrated).
- 4-6: Partial pass (respects restrictions but lacks timing, portions, or complete macronutrient metrics).
- 7-8: Good (meets all major requirements and restrictions with minor formatting/precision lapses).
- 9-10: Perfect production quality (meets 100% of criteria, clear meal timings, exact gram portions, zero restriction violations).
"""
    
    raw_eval = prompt_gemini(
        eval_prompt,
        system_instruction="You are a rigorous, calibrated AI evaluation judge specializing in athletic nutrition.",
        temperature=0.0,
        response_schema=EvaluationResult
    )
    
    try:
        return json.loads(raw_eval.strip())
    except Exception:
        return {"score": 5.0, "reasoning": "Fallback parsing", "passed_criteria": [], "failed_criteria": []}

def run_gemini_benchmark(prompt_fn, name="Candidate"):
    """Execute benchmark across all test cases with Gemini."""
    results = []
    print(f"\n🚀 Running Gemini Benchmark: [{name}]...")
    for idx, test_case in enumerate(test_dataset, 1):
        output = prompt_fn(test_case["prompt_inputs"])
        grade = grade_solution_gemini(test_case, output)
        results.append({
            "test_case": test_case["scenario"],
            "score": grade.get("score", 0.0),
            "reasoning": grade.get("reasoning", ""),
            "output_preview": output[:180].replace("\n", " ") + "..."
        })
        print(f"  • Case {idx}/5 [{test_case['scenario'][:32]}...]: Score = {grade.get('score', 0):.1f}/10")
        
    avg_score = mean([r["score"] for r in results])
    pass_rate = (len([r for r in results if r["score"] >= 7.0]) / len(results)) * 100
    print(f"📊 Summary for [{name}]: Avg Score = {avg_score:.2f}/10 | Pass Rate (>=7.0) = {pass_rate:.1f}%")
    return {"name": name, "avg_score": avg_score, "pass_rate": pass_rate, "results": results}



---
## Step 3: Baseline Iteration 0 — The Vague Prompt
Just as with Claude, an unconstrained prompt produces generic advice lacking quantifiable sports metrics.



In [ ]:
# Candidate 0: Vague Prompt on Gemini
def prompt_v0_vague_gemini(inputs):
    prompt = f"""
    Generate a meal plan for an athlete.
    Height: {inputs['height']}
    Weight: {inputs['weight']}
    Goal: {inputs['goal']}
    Restrictions: {inputs['restrictions']}
    """
    return prompt_gemini(prompt)

gemini_results_v0 = run_gemini_benchmark(prompt_v0_vague_gemini, name="V0: Vague Baseline")



---
## Step 4: Iteration 1 — Being Clear & Direct (`system_instruction`)
On Gemini, the most effective way to establish a persona and directives is using `system_instruction` in `GenerateContentConfig`.



In [ ]:
# Candidate 1: Clear & Direct Prompt with system_instruction
def prompt_v1_clear_direct_gemini(inputs):
    system_instruction = "You are an elite sports performance dietitian specializing in athletic meal programming."
    
    prompt = f"""
    Generate a structured, single-day athletic meal plan designed specifically for the following athlete profile:
    
    Athlete Profile:
    - Height: {inputs['height']}
    - Weight: {inputs['weight']}
    - Primary Goal: {inputs['goal']}
    - Dietary Restrictions: {inputs['restrictions']}
    
    Your plan must directly support the athlete's primary goal while strictly avoiding any ingredients that violate their dietary restrictions.
    """
    return prompt_gemini(prompt, system_instruction=system_instruction)

gemini_results_v1 = run_gemini_benchmark(prompt_v1_clear_direct_gemini, name="V1: Clear & Direct")



---
## Step 5: Iteration 2 — Concrete Constraints & Metrics
Adding numerical guidelines (daily calories, macronutrient grams, clock timings, gram portions).



In [ ]:
# Candidate 2: Specific Constraints on Gemini
def prompt_v2_specific_constraints_gemini(inputs):
    system_instruction = "You are an elite sports performance dietitian specializing in athletic meal programming."
    
    prompt = f"""
    Generate a structured, single-day athletic meal plan designed specifically for the following athlete profile:
    
    Athlete Profile:
    - Height: {inputs['height']}
    - Weight: {inputs['weight']}
    - Primary Goal: {inputs['goal']}
    - Dietary Restrictions: {inputs['restrictions']}
    
    Follow these specific guidelines:
    1. State the calculated daily calorie target and macronutrient breakdown (protein, carbs, fat in grams).
    2. Divide the day into at least 4 distinct meals/snacks with exact clock times (e.g., 7:30 AM Breakfast).
    3. Specify exact portion sizes in grams (g) for every food ingredient.
    4. Adhere 100% strictly to all stated dietary restrictions; never suggest prohibited ingredients or common contaminants.
    5. Prioritize nutrient-dense, recovery-promoting whole foods tailored to the athlete's specific sport demands.
    """
    return prompt_gemini(prompt, system_instruction=system_instruction)

gemini_results_v2 = run_gemini_benchmark(prompt_v2_specific_constraints_gemini, name="V2: Specific Constraints")



---
## Step 6: Iteration 3 — Structured Delimiters for Gemini
Gemini models respond exceptionally well to Markdown section delimiters (`## Input Data`, `## Operational Rules`, `## Output Schema`).



In [ ]:
# Candidate 3: Delimited Sections on Gemini
def prompt_v3_structured_delimiters_gemini(inputs):
    system_instruction = "You are an elite sports performance dietitian specializing in athletic meal programming."
    
    prompt = f"""
    Your task is to generate a comprehensive, single-day athletic nutrition plan.
    
    ## Athlete Profile
    - Height: {inputs['height']}
    - Weight: {inputs['weight']}
    - Primary Goal: {inputs['goal']}
    - Dietary Restrictions: {inputs['restrictions']}
    
    ## Guidelines & Operational Rules
    1. Caloric Target: Calculate and state total daily calories appropriate for the athlete's body mass and training stimulus.
    2. Macronutrients: Provide total protein, carbohydrate, and fat targets in grams.
    3. Meal Schedule: Provide at least 4 distinct eating windows with specific clock timings.
    4. Portions: List every food item with exact portion weight in grams (g).
    5. Dietary Safety: Strict zero-tolerance for any ingredient violating the stated restrictions.
    
    ## Output Format Required
    # Athlete Nutritional Overview
    - Daily Target: [X] kcal | Protein: [X]g | Carbohydrates: [X]g | Fat: [X]g
    
    # Scheduled Daily Meal Plan
    - [Time] [Meal Name]: [Portion g] [Food Item] ...
    """
    return prompt_gemini(prompt, system_instruction=system_instruction)

gemini_results_v3 = run_gemini_benchmark(prompt_v3_structured_delimiters_gemini, name="V3: Structured Delimiters")



---
## Step 7: Iteration 4 — Few-Shot In-Context Exemplars
Supplying an exact reference example anchors Gemini's formatting density and ensures sub-meal macro breakdowns.



In [ ]:
# Candidate 4: Few-Shot In-Context Learning on Gemini
def prompt_v4_few_shot_gemini(inputs):
    system_instruction = """You are an elite sports performance dietitian.
Generate precise, scientifically sound athletic nutrition plans following the exact formatting standards demonstrated in reference examples."""

    prompt = f"""
    Your task is to generate a comprehensive, single-day athletic nutrition plan.
    
    ## Reference Example
    
    ### Example Athlete Input:
    Height: 175 cm | Weight: 72 kg | Goal: High-volume endurance cycling | Restrictions: Dairy-free
    
    ### Ideal Output Structure:
    # Athlete Nutritional Overview
    - Daily Target: 2,850 kcal | Protein: 145g | Carbohydrates: 410g | Fat: 70g
    - Dietary Protocol: Strictly dairy-free (calcium fortified via plant sources)

    # Scheduled Daily Meal Plan

    ### 07:00 AM — Pre-Training Breakfast
    - 100g rolled oats cooked in 300ml unsweetened almond milk
    - 120g sliced banana
    - 30g natural peanut butter
    - 25g plant-based pea/rice protein powder
    *Meal Macros: 650 kcal | 38g Protein | 88g Carbs | 18g Fat*

    ### 12:30 PM — Post-Workout Recovery Lunch
    - 150g grilled chicken breast
    - 250g steamed jasmine rice
    - 150g steamed broccoli florets
    - 15ml extra virgin olive oil
    *Meal Macros: 740 kcal | 52g Protein | 80g Carbs | 22g Fat*

    ### 04:00 PM — Afternoon Fuel Snack
    - 60g whole wheat toast
    - 50g mashed avocado
    - 2 hard-boiled eggs
    *Meal Macros: 380 kcal | 18g Protein | 32g Carbs | 19g Fat*

    ### 07:30 PM — Nutrient Restoration Dinner
    - 180g baked Atlantic salmon
    - 220g baked sweet potato
    - 120g roasted asparagus
    *Meal Macros: 680 kcal | 46g Protein | 55g Carbs | 24g Fat*
    
    ---
    
    ## Target Athlete Profile
    - Height: {inputs['height']}
    - Weight: {inputs['weight']}
    - Primary Goal: {inputs['goal']}
    - Dietary Restrictions: {inputs['restrictions']}
    
    ## Guidelines & Operational Rules
    1. Caloric Target: Calculate and state total daily calories calibrated for the athlete.
    2. Macronutrients: Provide daily totals and per-meal breakdowns in grams.
    3. Meal Schedule: Provide at least 4 distinct eating windows with specific clock timings.
    4. Portions: List every food item with exact portion weight in grams (g).
    5. Dietary Safety: Strict zero-tolerance for any ingredient violating the stated restrictions.
    """
    return prompt_gemini(prompt, system_instruction=system_instruction)

gemini_results_v4 = run_gemini_benchmark(prompt_v4_few_shot_gemini, name="V4: Structured + Few-Shot")



---
## Step 8: Multi-Iteration Gemini Benchmark Comparison Table



In [ ]:
# Benchmark Results Summary Table: Google Gemini
all_gemini_benchmarks = [gemini_results_v0, gemini_results_v1, gemini_results_v2, gemini_results_v3, gemini_results_v4]

print("=" * 75)
print(f"{'Iteration / Gemini Strategy':<30} | {'Avg Score':<12} | {'Pass Rate (>=7.0)':<18}")
print("=" * 75)

for b in all_gemini_benchmarks:
    print(f"{b['name']:<30} | {b['avg_score']:>6.2f} / 10 | {b['pass_rate']:>12.1f}%")
print("=" * 75)

gemini_score_gain = gemini_results_v4['avg_score'] - gemini_results_v0['avg_score']
gemini_pass_gain = gemini_results_v4['pass_rate'] - gemini_results_v0['pass_rate']
print(f"\n🏆 Cumulative Gemini Engineering Gain: +{gemini_score_gain:.2f} points ({gemini_pass_gain:+.1f}% pass rate)")



---
## 🎓 Dual-API Comparison: Claude vs. Gemini
| Dimension | Anthropic Claude 3.5 | Google Gemini 2.5 Flash |
| :--- | :--- | :--- |
| **Persona Steering** | In-prompt system role or top-level `system` string | `system_instruction` in `types.GenerateContentConfig` |
| **Structural Delimiters** | XML Tags (`<context>`, `<document>`, `<rules>`) | Markdown Headings (`## Section`) or Tagged Blocks |
| **Structured JSON** | Assistant prefilling (`{"role": "assistant", "content": "{"}`) | Native Pydantic Schemas (`response_schema=BaseModel`) |
| **Few-Shot Steering** | `<example>` blocks with sample input/output | Demonstration turns or inline reference sections |
| **Edge-Case Nuance** | Exceptional adherence to negative constraints | Extremely fast token generation & schema enforcement |

